# EDA PortWatch — Ормуз и Баб-эль-Мандеб · чекпоинт 2

**Цель:** первичный анализ ежедневного трафика судов IMF PortWatch и соответствующих месячных признаков для дальнейшего объединения с ценами из World Bank Pink Sheet.

Что покрывает EDA: статистики всех показателей, пропуски, проверки качества, выбросы, распределения, сегменты судов, тренды и сезонность, корреляционные матрицы и pairplot, выводы. Публикуемые графики и статистики строятся **только на train**.

**Фиксированное разбиение до EDA:** train = 2019–2024; validation = 2025; test = 2026+. Валидация и тест не участвуют в настройке фильтров, нормировок, выборе признаков или интерпретации распределений. Месячная таблица для объединения с Pink Sheet — 2015-04 … 2026-09; до 2019 г. наблюдений нет.

**Важно:** `n_*` — число проходов, `capacity_*` — оценочная вместимость судов, **не масса карбамида**. Источник: https://portwatch.imf.org/

## 0. Установка библиотек

Ячейка устанавливает **только отсутствующие** пакеты в текущее окружение Jupyter; это не меняет другие проекты. Необходим установленный Python 3.10+ и Jupyter/VS Code.

In [ ]:
import sys, subprocess, importlib.util
NEEDED = {"pandas":"pandas", "numpy":"numpy", "matplotlib":"matplotlib", "seaborn":"seaborn"}
missing = [pip_name for import_name, pip_name in NEEDED.items() if importlib.util.find_spec(import_name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
print("Библиотеки готовы.")

## 1. Чтение и разделение по времени — до EDA

In [ ]:
from pathlib import Path
import calendar
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", font_scale=0.9)
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data/raw/portwatch_hormuz_bab_daily.csv").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Не найден data/raw/portwatch_hormuz_bab_daily.csv. Откройте ноутбук из папки репозитория.")
RAW = ROOT / "data/raw/portwatch_hormuz_bab_daily.csv"
MONTHLY = ROOT / "data/processed/portwatch_hormuz_bab_monthly.csv"
REPORT_DIR = ROOT / "reports/portwatch"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAW, encoding="utf-8-sig", parse_dates=["date"])
NUM_COLS = [c for c in df.columns if c.startswith("n_") or c.startswith("capacity")]
COUNT_COLS = [c for c in NUM_COLS if c.startswith("n_")]
for col in NUM_COLS:
    df[col] = pd.to_numeric(df[col], errors="coerce")
ROUTES = {"chokepoint4": "Баб-эль-Мандеб", "chokepoint6": "Ормуз"}
assert set(df["portid"].unique()) == set(ROUTES), "Проверить portid"
assert df["date"].notna().all(), "Есть нераспознанные даты"

TRAIN_END, VAL_END = pd.Timestamp("2025-01-01"), pd.Timestamp("2026-01-01")
train = df.loc[df.date < TRAIN_END].copy()
validation = df.loc[(df.date >= TRAIN_END) & (df.date < VAL_END)]
test = df.loc[df.date >= VAL_END]
train["route"] = train["portid"].map(ROUTES)
print(f"Датасет: train={len(train)} строк, validation={len(validation)} строк, test={len(test)} строк")
print("Период train:", train.date.min().date(), "—", train.date.max().date())
print("Дальнейший анализ — только train. Тест отложен.")

## 2. Качество данных: дубликаты, календарные и числовые пропуски, ошибки

Для целочисленных счётчиков проверяем точное равенство сумм. Для вместимости показываем малые расхождения отдельно: они могут быть следствием округления в опубликованных полях, но причина не установлена. Небольшое отклонение **не равно** доказанной ошибке записи.

In [ ]:
# Проверки только train
quality = []
missing_all = []
zero_all = []
for pid, route in ROUTES.items():
    g = train.loc[train.portid == pid].sort_values("date")
    full_dates = pd.date_range(g.date.min(), g.date.max(), freq="D")
    dup = int(g.duplicated(["date", "portid"]).sum())
    missing_dates = len(set(full_dates) - set(g.date))
    neg = int((g[NUM_COLS] < 0).sum().sum())
    frac = int(((g[COUNT_COLS].dropna() % 1) != 0).sum().sum())
    cargo_delta = g["capacity_cargo"] - g[["capacity_container","capacity_dry_bulk","capacity_general_cargo","capacity_roro"]].sum(axis=1)
    total_delta = g["capacity"] - (g["capacity_cargo"] + g["capacity_tanker"])
    sums = {
        "n_cargo != types": int((g["n_cargo"] != g[["n_container","n_dry_bulk","n_general_cargo","n_roro"]].sum(axis=1)).sum()),
        "n_total != cargo+tanker": int((g["n_total"] != g["n_cargo"]+g["n_tanker"]).sum()),
        "capacity_cargo_small_deltas": int(cargo_delta.ne(0).sum()),
        "capacity_cargo_abs_delta_max": float(cargo_delta.abs().max()),
        "capacity_cargo_delta_gt3": int(cargo_delta.abs().gt(3).sum()),
        "capacity_small_deltas": int(total_delta.ne(0).sum()),
        "capacity_abs_delta_max": float(total_delta.abs().max()),
        "capacity_delta_gt1": int(total_delta.abs().gt(1).sum()),
    }
    quality.append({"route":route,"days":len(g),"duplicate_dates":dup,"missing_calendar_days":missing_dates,
                    "negative_values":neg,"fractional_counts":frac, **sums})
    for c in NUM_COLS:
        missing_all.append({"route":route,"feature":c,"missing":int(g[c].isna().sum()),"missing_%":round(100*g[c].isna().mean(),2)})
        zero_all.append({"route":route,"feature":c,"zero_days":int(g[c].eq(0).sum()),"zero_%":round(100*g[c].eq(0).mean(),2)})
quality_df = pd.DataFrame(quality)
missing_df = pd.DataFrame(missing_all)
zero_df = pd.DataFrame(zero_all)
display(quality_df, missing_df, zero_df)
quality_df.to_csv(REPORT_DIR / "quality_checks_train.csv",index=False, encoding="utf-8-sig")
missing_df.to_csv(REPORT_DIR / "missing_values_train.csv",index=False, encoding="utf-8-sig")
zero_df.to_csv(REPORT_DIR / "zero_values_train.csv",index=False, encoding="utf-8-sig")

## 3. Описательная статистика **всех** числовых полей

In [ ]:
rows=[]
for pid, route in ROUTES.items():
    g = train.loc[train.portid == pid]
    desc = g[NUM_COLS].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T
    desc.insert(0, "route", route)
    desc.insert(1, "feature", desc.index)
    rows.append(desc.reset_index(drop=True))
stats = pd.concat(rows, ignore_index=True)
display(stats.round(2))
stats.to_csv(REPORT_DIR / "statistics_train.csv", index=False, encoding="utf-8-sig")

## 4. Подозрительные нули (не обязательно пропуски)

`capacity_dry_bulk = 0` при `n_dry_bulk > 0` — подозрительное сочетание: судно прошло, но вместимость неизвестна или записана как 0. **Автоматически не заменяем**.

In [ ]:
sus=[]
for pid, route in ROUTES.items():
    g = train.loc[train.portid == pid]
    for ship_type in ["container", "dry_bulk", "general_cargo", "roro", "tanker"]:
        z = g[(g[f"n_{ship_type}"] > 0) & (g[f"capacity_{ship_type}"] == 0)]
        for _,r in z.iterrows():
            sus.append({"route":route,"date":r.date.date().isoformat(),"ship_type":ship_type,
                        "ships":int(r[f"n_{ship_type}"]),"capacity":float(r[f"capacity_{ship_type}"])})
suspicious_zeros = pd.DataFrame(sus,columns=["route","date","ship_type","ships","capacity"])
suspicious_zeros.to_csv(REPORT_DIR / "suspicious_zero_capacity_train.csv",index=False, encoding="utf-8-sig")
print("Подозрительных нулей вместимости по типам судов:")
display(suspicious_zeros.groupby(["route","ship_type"]).size().rename("days").reset_index() if len(suspicious_zeros) else "Нет")
print("Сухогрузы:")
display(suspicious_zeros.loc[suspicious_zeros.ship_type=="dry_bulk"])

## 5. Выбросы (1.5×IQR), только train

IQR выявляет кандидатов на выбросы, **но не доказывает ошибку**. При логистическом кризисе необычно низкий трафик может быть самым интересным наблюдением.

In [ ]:
core = ["n_dry_bulk","capacity_dry_bulk","n_total","capacity"]
records=[]
for pid,route in ROUTES.items():
    g=train.loc[train.portid == pid]
    for col in core:
        q1,q3 = g[col].quantile([0.25,0.75]); iqr=q3-q1
        lower,upper = q1-1.5*iqr, q3+1.5*iqr
        bad = g[(g[col] < lower) | (g[col] > upper)]
        for _,r in bad.iterrows():
            records.append({"route":route,"date":r.date.date().isoformat(),"feature":col,
                            "value":float(r[col]),"lower_bound":float(lower),"upper_bound":float(upper)})
outliers = pd.DataFrame(records, columns=["route","date","feature","value","lower_bound","upper_bound"])
outliers.to_csv(REPORT_DIR / "outliers_iqr_train.csv", index=False, encoding="utf-8-sig")
display(outliers.groupby(["route","feature"]).size().rename("n_outliers").unstack(fill_value=0))

## 6. Временные тренды (train): сухогрузы, общие суда, вместимость

In [ ]:
train_m = (train.assign(date=train.date.dt.to_period("M").dt.to_timestamp())
           .groupby(["date","route"])[["n_dry_bulk","capacity_dry_bulk","n_total"]]
           .mean().reset_index())
fig,axes=plt.subplots(3,1,figsize=(12,10),sharex=True)
for i, col in enumerate(["n_dry_bulk","capacity_dry_bulk","n_total"]):
    sns.lineplot(data=train_m,x="date",y=col,hue="route",ax=axes[i],marker="o",markersize=2)
    axes[i].set_ylabel({"n_dry_bulk":"Сухогрузы / день","capacity_dry_bulk":"Вместимость сухогрузов / день","n_total":"Все суда / день"}[col]);axes[i].set_xlabel("")
fig.suptitle("PortWatch: месячные средние на обучающем периоде (2019–2024)")
fig.tight_layout();fig.savefig(REPORT_DIR / "traffic_trends_train.png",dpi=150,bbox_inches="tight");plt.show()

## 7. Структура судов и сезонность (train)

Доли рассчитываются от числа **всех** проходов; tanker и cargo не суммируются с уже включёнными в cargo подтипами дважды. Сезонность показана описательно: смешение с кризисными периодами может создавать ложные «месячные закономерности».

In [ ]:
plot_cols=["n_container","n_dry_bulk","n_general_cargo","n_roro","n_tanker"]
shares=(train.groupby("route")[plot_cols].sum())
shares=shares.div(shares.sum(axis=1),axis=0)*100
ax=shares.plot(kind="barh",stacked=True,figsize=(10,4));ax.set(xlabel="Доля проходов по числу судов, %",ylabel="",title="Структура трафика по типу судов (train)")
plt.legend(bbox_to_anchor=(1.02,1), loc="upper left");plt.tight_layout();plt.savefig(REPORT_DIR/"ship_types_share_train.png",dpi=150,bbox_inches="tight");plt.show()

month_profile=train.assign(month=train.date.dt.month).groupby(["route","month"],as_index=False)["n_dry_bulk"].mean()
fig,ax=plt.subplots(figsize=(10,4));sns.lineplot(data=month_profile,x="month",y="n_dry_bulk",hue="route",marker="o",ax=ax)
ax.set(xticks=list(range(1,13)),xlabel="Месяц",ylabel="Сухогрузы / день",title="Сезонный профиль 2019–2024 (описательно)")
fig.tight_layout();fig.savefig(REPORT_DIR/"seasonality_train.png",dpi=150,bbox_inches="tight");plt.show()

## 8. Распределения и экстремумы (train)

In [ ]:
fig,axes=plt.subplots(2,2,figsize=(12,8))
for ax, col in zip(axes.flat,["n_dry_bulk","n_total","capacity_dry_bulk","capacity"]):
    sns.histplot(data=train,x=col,hue="route",stat="density",common_norm=False,alpha=0.4,ax=ax,bins=35)
    ax.set_title(col)
fig.suptitle("Распределения числовых показателей, train",y=1.01)
fig.tight_layout();fig.savefig(REPORT_DIR/"distributions_train.png",dpi=150,bbox_inches="tight");plt.show()

## 9. Корреляционная матрица (train, месячная агрегация)

Месячные средние вычислены **из train**, чтобы не использовать тест. Корреляция — не причинность. Некоторые суммы (`n_total`,`n_cargo`,`capacity`) механически связаны с типами судов. Корреляция **с ценой карбамида** требует совместного датасета Pink Sheet, которого здесь нет.

In [ ]:
wide=(train_m.pivot(index="date",columns="route",values=["n_dry_bulk","capacity_dry_bulk","n_total"])
      .sort_index(axis=1))
wide.columns=[f"{route}: {field}" for field, route in wide.columns]
pearson=wide.corr(method="pearson")
spearman=wide.corr(method="spearman")
fig,axes=plt.subplots(1,2,figsize=(17,7))
sns.heatmap(pearson,annot=True,fmt=".2f",cmap="vlag",vmin=-1,vmax=1,ax=axes[0],square=True)
sns.heatmap(spearman,annot=True,fmt=".2f",cmap="vlag",vmin=-1,vmax=1,ax=axes[1],square=True)
axes[0].set_title("Pearson");axes[1].set_title("Spearman")
fig.tight_layout();fig.savefig(REPORT_DIR/"correlations_train.png",dpi=140,bbox_inches="tight");plt.show()
pearson.to_csv(REPORT_DIR/"pearson_correlation_train.csv",encoding="utf-8-sig")
spearman.to_csv(REPORT_DIR/"spearman_correlation_train.csv",encoding="utf-8-sig")

## 10. Pairplot — парные зависимости месячных признаков (train)

Берём четыре ключевых поля, чтобы график оставался читаемым. Разная размерность судов и вместимости требует осторожности в сравнении наклонов.

In [ ]:
small=(train_m.pivot(index="date", columns="route", values=["n_dry_bulk","capacity_dry_bulk"])
       .sort_index(axis=1))
short_route = {"Баб-эль-Мандеб":"Mandeb", "Ормуз":"Hormuz"}
short_field = {"n_dry_bulk":"bulk / day", "capacity_dry_bulk":"capacity / day"}
small.columns=[f"{short_route[route]} {short_field[field]}" for field,route in small.columns]
small=small.dropna()
g=sns.pairplot(small, corner=True, diag_kind="hist",plot_kws={"alpha":0.65,"s":25,"color":"C0"},diag_kws={"color":"C0"})
g.fig.suptitle("Парные графики: сухогрузы и вместимость, train (месячные средние)", y=1.02)
g.fig.savefig(REPORT_DIR/"pairplot_train.png",dpi=140,bbox_inches="tight")
plt.show()

## 11. Изменение трафика по годам и конкретные выводы (train)

In [ ]:
annual=train.assign(year=train.date.dt.year).groupby(["route","year"],as_index=False)[["n_dry_bulk","capacity_dry_bulk","n_total"]].mean()
display(annual.round(2))
annual.to_csv(REPORT_DIR/"annual_means_train.csv",index=False,encoding="utf-8-sig")
for route in ROUTES.values():
    one=annual[annual.route==route].set_index("year")
    change=(one.loc[2024, ["n_dry_bulk","capacity_dry_bulk","n_total"]]/
            one.loc[2023, ["n_dry_bulk","capacity_dry_bulk","n_total"]]-1)*100
    print(route,"2024 к 2023 (%, среднесуточные значения):",change.round(1).to_dict())